# TeCo sentiment: unseen readers

**CPU → Run all. Use Zyion for both Drive prompts.** This is the follow-up to `teco_other_targets`: the same frozen labels, saved features, model search, controls and random seed, now holding out readers for all three sentiment targets.

| Targets | Positive/negative; neutral/nonneutral; positive/negative/neutral |
|---|---|
| Data | TeCo task 1 only. Original completed target rosters, including original pooled-neutral definition. |
| Averaging | Individual trials (1), exact10, exact40, Max40: 12 target/averaging conditions. |
| Splits | Three unseen-reader outer folds; two unseen-reader inner validation folds. Words/sentences may overlap. |
| Models | Screen inputs with logistic C=1, then top3 inputs across nine settings: logistic C=.1/1/10, shrinkage LDA, approximate RBF, Extra Trees, Random Forest, MLP, kNN. |
| Outputs | Fixed EEG, validation-selected EEG, pre-onset, word-properties-only and dummy. Balanced accuracy and macro-F1, predictions, selected models, exact memberships, unavailable conditions. |

**No EEG loading, extraction, relabeling, or feature preparation.** Requires completed `teco_other_targets`, original classification feature caches, original sentiment prediction rosters and the earlier completed TeCo sweep's binary feature cache. The script verifies source identities and exact event IDs/labels against the completed reference. It stops on changed/missing inputs rather than silently changing the experiment.

Fixed EEG remains **all-channel224–304ms means for positive/negative**, and **48regional mean-voltage features (six regions×eight100ms bins,0–800ms) for the other targets**. Selected inputs include15regional representations; binary also reuses20advanced saved representations. No raw waveform CNN is included: the small MLP is one of the existing feature-based models.

Split first, then average within reader and known target class, separately in each partition. Exact sizes discard remainders; Max40 retains them. Averages use known test labels, so their scores describe class-labelled averages, not single-word deployment. Keep the existing1000ms train/heldout overlap purge within reader/sentence, also inside validation; reader holdout itself prevents shared reader recordings across partitions. Unavailable conditions are reported without relaxing the selection rules.

The metadata control uses word length, relative position, sentence length and control-sentence flag. This notebook does not fit paper-style regression cleaning and does not overwrite earlier familiar-identity results.

**Storage:** at most50MB new Drive output in `Thesis/Results/word_sentiment_eeg/teco_sentiment_unseen_readers/`. No new persistent EEG/features/model bank. Completed fits resume. CPU runtime has not been benchmarked; the earlier larger other-target run took about23minutes, but do not treat that as a promise. Logs show reader X/12, target X/3, fold and averaging size, with30-second heartbeat updates.

Results remain exploratory on an already examined dataset; no significance test is claimed. The final table below shows all12 planned conditions, including any incomplete/unavailable cases, plus the fold-specific selected input/model for updating the comparison table later.


In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'CPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/teco_sentiment_unseen_readers'
REPO=Path('/content/word-sentiment-eeg-teco-sentiment-readers')
CODE_REVISION='b67c180d4765f1299471fd81596d3baa1151c785'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
# Temporary cloud-only scientific environment: no base-runtime upgrades/restart.
# BOOTSTRAP_START: also exercised verbatim in cloud CI.
# Use a file-rendering backend in subprocesses, not Colab's unavailable inline plugin.
os.environ['MPLBACKEND']='Agg'
ENV=Path('/content/word-eeg-paper-classification-python')
PYTHON=ENV/'bin/python'
def setup_command(command):
    print('SETUP:', ' '.join(map(str,command)), flush=True)
    result=subprocess.run(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    if result.stdout:print(result.stdout,flush=True)
    if result.returncode:
        raise RuntimeError('Scientific environment setup failed; the full diagnostic is printed above. No EEG processing started.')
# Colab may lack ensurepip. This also repairs a partially created environment.
setup_command([sys.executable,'-m','venv','--without-pip',str(ENV)])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'install',
               '--only-binary=:all:','-r',str(REPO/'requirements-paper-classification.txt')])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'check'])
setup_command([str(PYTHON),'-c',
    'import numpy,pandas,scipy,sklearn,matplotlib; '
    'print("Scientific environment ready:",numpy.__version__,pandas.__version__,scipy.__version__,sklearn.__version__,matplotlib.__version__)'])
# BOOTSTRAP_END
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# Verify account quota before loading any EEG. Use Zyion for both prompts.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated quota account:',email)
if email!='zyionygvmutch39@gmail.com':
    raise RuntimeError('Choose your Zyion account for Drive authentication and mounting; it contains the saved trials.')
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a storage limit; cannot verify free space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,50_000_000-used):
    raise RuntimeError('Reserve at most 50 MB for results. This notebook reads existing feature caches only; no EEG shards or extraction.')
print(f'Free space: {FREE_DRIVE_BYTES/1e9:.2f} GB; new output cap: 50 MB.')
print('New output folder:',OUT)


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[str(PYTHON),'-u',str(REPO/'scripts/teco_sentiment_unseen_readers.py'),'--thesis-root',str(THESIS),'--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='TeCo sentiment: unseen readers, three targets and four averaging sizes')


In [ ]:
import hashlib, gzip
from IPython.display import display, Markdown, Image
state=json.loads((OUT/'complete.json').read_text())
assert state['status']=='completed'
for name,sha in state['checksums'].items():
    if hashlib.sha256((OUT/name).read_bytes()).hexdigest()!=sha:raise RuntimeError('Checksum failed: '+name)
summary=pd.read_csv(OUT/'summary.csv',dtype={'size':str})
rows=json.loads(gzip.decompress((OUT/'outer_results.json.gz').read_bytes()))
report=[]
variants=['fixed_eeg','selected_eeg','pre_control','metadata_control','dummy']
for target in ['sentiment_binary','neutral_nonneutral','sentiment_three_class']:
    for size in ['1','10','40','max40']:
        g=summary[(summary.target==target)&summary['size'].eq(size)]
        rec=dict(target=target,holdout='Unseen readers',averaging=size)
        for variant in variants:
            r=g[g.variant.eq(variant)]
            rec[variant+' BA / macro-F1 (%)']=('Unavailable' if r.empty else 'Incomplete' if int(r.iloc[0].completed_folds)!=3 else f'{r.iloc[0].balanced_accuracy*100:.1f} / {r.iloc[0].macro_f1*100:.1f}')
        chosen=sorted([r for r in rows if r['target']==target and str(r['size'])==size and r['variant']=='selected_eeg'],key=lambda r:r['fold'])
        rec['Selected input/model per fold']='; '.join(f"Fold{r['fold']+1}: {r['input']} → {r['model']}" for r in chosen) or 'Unavailable'
        report.append(rec)
display(Markdown('## Completed run: all 12 planned conditions'))
with pd.option_context('display.max_rows',100,'display.max_columns',20,'display.max_colwidth',160):
    display(pd.DataFrame(report))
display(Image(filename=str(OUT/'overview.png')))
print('BA chance:50% binary,33.3% three-class. Averaged results use known labels; no significance test.')


In [ ]:
display(Markdown('## Target definitions and sample counts'))
for path in sorted((OUT/'targets').glob('*/definition.json')):
    value=json.loads(path.read_text());print(path.parent.name, json.dumps(value,ensure_ascii=False))
display(Markdown('## Chosen configurations — compare selected_eeg with fixed_eeg'))
rows=json.loads(gzip.decompress((OUT/'outer_results.json.gz').read_bytes()))
chosen=pd.DataFrame([{k:r.get(k) for k in ['target','mode','fold','size','variant','input','model','status','balanced_accuracy','test_examples','actual_group_min','actual_group_max','reason']} for r in rows])
with pd.option_context('display.max_rows',300):display(chosen[chosen.variant.isin(['selected_eeg','fixed_eeg'])].round(3))
display(Markdown('## Unavailable conditions'))
missing=pd.read_csv(OUT/'unavailable.csv')
with pd.option_context('display.max_rows',200):display(missing)
print('Saved results:', OUT)
print('summary.csv = final scores; development.csv.gz = validation search; outer_results.json.gz = predictions; fits/ = resumable compressed checkpoints and exact event memberships.')
